# HTML 웹앱 임베드 예시

코드 셀을 실행하면 출력 영역에 웹앱이 나타난다. 마크다운 셀에서는 `<script>`가 실행되지 않으므로 반드시 **코드 셀 출력**으로 넣는다.

| 예시 | 방식 |
|---|---|
| 1 | HTML·JS를 셀 안에 직접 작성 |
| 2 | HTML 파일을 읽어 와서 표시 |
| 3 | Python 데이터를 JS로 넘겨 차트 그리기 |
| 4 | iframe으로 격리해 같은 앱을 여러 번 띄우기 |
| 5 | 단계별로 넘겨 보는 흐름도 |
| 6 | PDF 교안을 슬라이드 뷰어로 보기 |
| 7 | PPTX를 슬라이드 뷰어로 보기 |

보조 파일은 `lessons/assets/`에 있다. 코드 셀의 경로는 노트북이 있는 `lessons/` 기준이고, 명령은 프로젝트 루트 기준이다.

## 1. 인라인

In [ ]:
from IPython.display import HTML, display

# HTML·JavaScript를 문자열로 노트북 안에 직접 넣는다.
display(
    HTML("""
<div style="font-family:sans-serif;padding:12px;border:1px solid #ccc;border-radius:8px;max-width:320px">
  <h3 style="margin:0 0 8px">인라인 카운터</h3>
  <button id="inline-btn">+1</button>
  <span id="inline-count">0</span>
</div>
<script>
  let inlineCount = 0;
  document.getElementById("inline-btn").onclick = () => {
    inlineCount += 1;
    document.getElementById("inline-count").textContent = inlineCount;
  };
</script>
""")
)

## 2. 파일 참조

`HTML(filename=...)`은 셀을 실행할 때 파일 내용을 읽어 온다. 파일을 고친 뒤 이 셀만 다시 실행하면 바뀐 앱이 나온다.

In [ ]:
from IPython.display import HTML, display

display(HTML(filename="assets/counter_app.html"))

## 3. Python 데이터 → JavaScript

Python 값을 `json.dumps`로 문자열로 만들어 HTML 안에 끼워 넣는다. 막대를 클릭하면 값이 표시된다.

In [ ]:
import json
from IPython.display import HTML, display

# Python 쪽 데이터. 이 값을 바꾸고 다시 실행하면 차트가 바뀐다.
scores = {"M1 과정": 4.63, "M1 강사": 4.62, "M2 과정": 4.64, "M2 강사": 4.71}

display(
    HTML(f"""
<div id="chart-app" style="font-family:sans-serif;max-width:420px">
  <h3 style="margin:0 0 8px">11차수 만족도 (5점 만점)</h3>
  <div data-role="bars"></div>
  <p data-role="picked" style="color:#555">막대를 클릭해 보세요.</p>
</div>
<script>
(() => {{
  const data = {json.dumps(scores, ensure_ascii=False)};
  const root = document.getElementById("chart-app");
  const bars = root.querySelector('[data-role="bars"]');
  for (const [label, value] of Object.entries(data)) {{
    const row = document.createElement("div");
    row.style.cssText = "display:flex;align-items:center;gap:6px;margin:4px 0;cursor:pointer";
    row.innerHTML = `<span style="width:70px">${{label}}</span>
      <span style="height:18px;width:${{value / 5 * 300}}px;background:#1f6feb;border-radius:3px"></span>`;
    row.onclick = () => root.querySelector('[data-role="picked"]').textContent = `${{label}}: ${{value}}점`;
    bars.appendChild(row);
  }}
}})();
</script>
""")
)

## 4. iframe으로 격리하기

VS Code는 한 노트북의 출력을 같은 화면에 그린다. 그래서 같은 `id`를 쓰는 앱을 두 번 띄우면 서로 간섭한다. `iframe`의 `srcdoc`에 넣으면 각 앱이 따로 돈다.

In [ ]:
import html
from pathlib import Path
from IPython.display import HTML, display

app = Path("assets/counter_app.html").read_text(encoding="utf-8")


def isolated(page: str, height: int = 170) -> HTML:
    """HTML 한 덩어리를 독립된 iframe 안에 넣는다."""
    # <div>로 감싸면 IPython의 "IFrame을 쓰라"는 경고가 나오지 않는다.
    return HTML(
        f'<div><iframe srcdoc="{html.escape(page)}" '
        f'style="width:360px;height:{height}px;border:none"></iframe></div>'
    )


# 같은 앱을 두 번 띄워도 각각 따로 동작한다.
display(isolated(app), isolated(app))

## 5. 단계별 흐름도

버튼을 누를 때마다 에이전트 루프의 다음 단계가 강조된다. 설명 순서에 맞춰 넘기며 보여 줄 때 쓴다.

In [ ]:
from IPython.display import HTML, display

display(
    HTML("""
<div id="loop-app" style="font-family:sans-serif;max-width:560px">
  <svg viewBox="0 0 560 150" style="width:100%">
    <defs><marker id="loop-arrow" viewBox="0 0 10 10" refX="10" refY="5" markerWidth="6" markerHeight="6" orient="auto">
      <path d="M0,0 L10,5 L0,10 z" fill="#888"/></marker></defs>
    <g data-step="0"><rect x="10"  y="40" width="110" height="50" rx="8"/><text x="65"  y="70" text-anchor="middle">사용자 요청</text></g>
    <g data-step="1"><rect x="160" y="40" width="110" height="50" rx="8"/><text x="215" y="70" text-anchor="middle">LLM 판단</text></g>
    <g data-step="2"><rect x="310" y="40" width="110" height="50" rx="8"/><text x="365" y="70" text-anchor="middle">도구 실행</text></g>
    <g data-step="3"><rect x="440" y="40" width="110" height="50" rx="8"/><text x="495" y="70" text-anchor="middle">최종 답변</text></g>
    <line x1="120" y1="65" x2="160" y2="65" stroke="#888" marker-end="url(#loop-arrow)"/>
    <line x1="270" y1="60" x2="310" y2="60" stroke="#888" marker-end="url(#loop-arrow)"/>
    <path d="M365,90 Q290,140 215,90" fill="none" stroke="#888" marker-end="url(#loop-arrow)"/>
    <text x="290" y="140" text-anchor="middle" font-size="12" fill="#888">결과를 다시 판단</text>
    <line x1="270" y1="45" x2="440" y2="45" stroke="#888" stroke-dasharray="4" marker-end="url(#loop-arrow)"/>
  </svg>
  <button data-act="next">다음 단계</button>
  <span data-role="desc" style="margin-left:8px"></span>
</div>
<script>
(() => {
  const root = document.getElementById("loop-app");
  const order = [0, 1, 2, 1, 3];
  const desc = ["질문이 들어온다", "LLM이 도구가 필요한지 판단한다", "도구를 실행한다",
                "도구 결과를 보고 다시 판단한다", "더 필요 없으면 답을 낸다"];
  let k = 0;
  const paint = () => {
    root.querySelectorAll("g[data-step]").forEach(g => {
      const on = Number(g.dataset.step) === order[k];
      g.querySelector("rect").setAttribute("fill", on ? "#1f6feb" : "#eee");
      g.querySelector("text").setAttribute("fill", on ? "#fff" : "#333");
    });
    root.querySelector('[data-role="desc"]').textContent = `${k + 1}/${order.length} ${desc[k]}`;
  };
  root.querySelector('[data-act="next"]').onclick = () => { k = (k + 1) % order.length; paint(); };
  paint();
})();
</script>
""")
)

## 6. PDF 교안 슬라이드 뷰어

PDF 페이지를 이미지로 바꿔 슬라이드 묶음(JSON) 한 파일로 만들어 두고, `show_slides`로 보여 준다. 버튼, 썸네일, ← → 키로 넘긴다.

묶음은 아래 명령으로 만든다. `pdftoppm`(poppler)이 필요하다.

```
uv run python lessons/assets/build_slide_deck.py docs/raw/Chapter02/day01/3-1-course-one-review-and-advanced-direction.pdf lessons/assets/slides/3-1.json
```

In [ ]:
import sys

sys.path.insert(0, "../../course/Chapter01")  # 챕터 폴더의 assets 노트북 도구를 불러오기 위한 경로
from assets.widgets import show_slides

show_slides("3-1")

## 7. PPTX 슬라이드 뷰어

PPTX는 LibreOffice(`soffice`)로 PDF로 바꾼 뒤 6번과 같은 방식으로 만든다. 애니메이션·전환 효과는 사라진다. `pages`로 원하는 쪽만 고를 수 있다.

```
uv run python lessons/assets/build_slide_deck.py docs/raw/2026-09-03_커리큘럼_제안서.pptx lessons/assets/slides/curriculum_0903.json
```

In [ ]:
from assets.widgets import show_slides

show_slides("2026-09-03_커리큘럼_제안서", pages=[2, 3, 4, 5])